In [ ]:
"""
TDDFT absorption spectrum for Cr2O(NH3)8^4+ (septet ground state)
to compare against the TPSCI calculation in tpsci.jl.

Reference SCF: loaded from scf.fchk (PySCF checkpoint, RHF reference).
TDDFT:         CAM-B3LYP/def2-SVP on top of UKS(CAM-B3LYP) reference.
               CAM-B3LYP is preferred here because the relevant excited
               states have strong charge-transfer character (CT between
               the two Cr centres and the oxo bridge).

Usage:
    python tddft.py

Output:
    tddft_spectrum.png  — broadened spectrum overlaid with sticks
    tddft_roots.txt     — excitation energies (eV) and oscillator strengths
"""


In [4]:

import numpy as np
import h5py, json
import pyscf
from pyscf import gto, dft, tddft
from pyscf.tools import molden
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HA2EV = 27.2114
molecule = """
 Cr 0.82627800 -1.30446200 -0.96524300
 Cr -0.82625500 1.30449100 0.96521400
 O 0.00001100 0.00001500 -0.00001400
 N 2.71379100 -0.58517300 -0.32156700
 H 3.52017600 -0.87998300 -0.88101600
 H 2.98302700 -0.84252100 0.62848200
 H 2.75216700 0.43162900 -0.34666000
 N 0.90611400 -0.01191500 -2.64373800
 H 0.15767300 0.67336300 -2.59732700
 H 0.79520000 -0.45289000 -3.56150100
 H 1.76766300 0.52845600 -2.74461100
 N 0.74644200 -2.59700900 0.71325300
 H 0.86887400 -3.59499400 0.51496700
 H -0.15371900 -2.54555600 1.18538900
 H 1.44183900 -2.41670700 1.43797600
 N -1.06123500 -2.02375100 -1.60891800
 H -1.51982700 -2.67507800 -0.96956400
 H -1.07760500 -2.52098500 -2.50431600
 H -1.71705200 -1.25706600 -1.73635500
 N 1.78821400 -2.82312800 -2.08895700
 H 2.20721200 -2.51946500 -2.97231100
 H 1.19687500 -3.61071400 -2.36865500
 H 2.56853800 -3.28036700 -1.60828700
 N -2.71376800 0.58520300 0.32153900
 H -2.72742600 -0.43151600 0.28926300
 H -3.01495100 0.88961900 -0.60504900
 H -3.51124000 0.82996900 0.91689300
 N -0.90609100 0.01194400 2.64371000
 H -0.86251400 0.45862400 3.56440800
 H -0.11862500 -0.63055500 2.63771800
 H -1.73984000 -0.57546100 2.70589400
 N 1.06125800 2.02378000 1.60888900
 H 1.08580100 2.47872200 2.52630400
 H 1.50134100 2.71049000 0.99382500
 H 1.72819800 1.26071800 1.68834600
 N -0.74641900 2.59703800 -0.71328200
 H -0.92607700 3.58837700 -0.52509900
 H -1.40673500 2.37764300 -1.45939400
 H 0.17080800 2.59240800 -1.15542700
 N -1.78819100 2.82315700 2.08892800
 H -1.19783000 3.61531400 2.35756800
 H -2.19798000 2.52398300 2.97800300
 H -2.57540900 3.27269300 1.61233200
"""

basis = "def2-svp"
pymol = pyscf.gto.Mole(
        atom    =   molecule,
        symmetry=   True,
        spin    =   6, # number of unpaired electrons
        charge  =   4,
        basis   =   basis)


pymol.build()
print("symmetry: ",pymol.topgroup)
# mf = pyscf.scf.UHF(pymol).x2c()
mf = pyscf.scf.ROHF(pymol)
mf.verbose = 4
mf.conv_tol = 1e-8
mf.conv_tol_grad = 1e-5
mf.chkfile = "scf.fchk"
mf.init_guess = "sad"

mf.run(max_cycle=200)

print(" Hartree-Fock Energy: %12.8f" % mf.e_tot)

symmetry:  C1


******** <class 'pyscf.scf.rohf.ROHF'> ********
method = ROHF
initial guess = sad
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-08
SCF conv_tol_grad = 1e-05
SCF max_cycles = 200
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = scf.fchk
max_memory 4000 MB (current use 0 MB)
num. doubly occ = 73  num. singly occ = 6


init E= -2722.30649096234
  HOMO = -0.18089342944938  LUMO = -0.091235073235848
cycle= 1 E= -2721.10285451241  delta_E=  1.2  |g|= 1.19  |ddm|= 4.33
  HOMO = -0.484313365416211  LUMO = -0.390433229329979
cycle= 2 E= -2721.51059453132  delta_E= -0.408  |g|= 0.767  |ddm|= 1.34
  HOMO = -0.59726314322341  LUMO = -0.394693000098059
cycle= 3 E= -2721.64604178939  delta_E= -0.135  |g|= 0.273  |ddm|= 0.643
  HOMO = -0.658985859320109  LUMO = -0.39026425175696
cycle= 4 E= -2721.66337406942  delta_E= -0.0173  |g|= 0.102  |ddm|= 0.278
  HOMO = -0.629457238654633  LUMO = -0.388999756567826
cycle= 5 E= -2721.66747378762  delta_E= -0.0041  |g|= 0.0297  |ddm|= 0.123
  HOMO = -0.638725503835326  LUMO = -0.388775332209739
cycle= 6 E= -2721.66781991663  delta_E= -0.000346  |g|= 0.00503  |ddm|= 0.0409
  HOMO = -0.638891031522616  LUMO = -0.38879170727746
cycle= 7 E= -2721.66783324035  delta_E= -1.33e-05  |g|= 0.00265  |ddm|= 0.015
  HOMO = -0.639229284648857  LUMO = -0.388820079206705
cycle= 8 E= -2721.

In [5]:

# ── 2. UKS ground state ───────────────────────────────────────────────────────
# Start from the ROHF MO coefficients stored in the checkpoint and break
# symmetry by mixing the HOMO and LUMO of each spin channel slightly.
mf = dft.UKS(pymol)
mf.xc     = "CAM-B3LYP"
mf.grids.level = 3
mf.max_cycle   = 200
mf.conv_tol    = 1e-9
mf.chkfile     = "uks_camb3lyp.chk"

# Initialise from stored RHF MOs then break symmetry
with h5py.File("scf.fchk", "r") as f:
    mo_coeff_rhf = f["scf/mo_coeff"][()]
    mo_occ_rhf   = f["scf/mo_occ"][()]

# Build UKS initial guess: alpha gets extra spin-up, beta gets spin-down.
# pyscf's init_guess_by_chkfile is the cleanest route when available.
mf.init_guess = "chk"
mf.chkfile    = "scf.fchk"   # read the RHF MOs as starting point

mf.kernel()

if not mf.converged:
    print("WARNING: UKS did not converge — TDDFT results may be unreliable.")


converged SCF energy = -2727.70728425877  <S^2> = 12.029823  2S+1 = 7.0085158


In [6]:

# ── 3. TDDFT ─────────────────────────────────────────────────────────────────
nroots = 30   # compute 30 excited states; increase if needed to cover UV range

td = tddft.TDDFT(mf)
td.nstates  = nroots
td.conv_tol = 1e-6
td.kernel()


TD-SCF states [6, 7, 10, 11] not converged.
Excited State energies (eV)
[2.03077206 2.0342139  2.1704241  2.17350078 2.41205222 2.4137809
 2.7432573  2.74357914 3.07264268 3.07547722 4.02694767 4.05276098
 4.38987506 4.41860183 4.43305908 4.70553258 4.70914868 4.73336113
 4.73708916 4.80633048 5.04384614 5.04431301 5.07343655 5.07371159
 5.40029427 5.40163758 5.42639519 5.43960526 5.5147256  5.66332492]


(array([0.0746295 , 0.07475598, 0.07976162, 0.07987468, 0.08864128,
        0.08870481, 0.10081285, 0.10082467, 0.11291754, 0.1130217 ,
        0.1479876 , 0.14893622, 0.16132493, 0.16238062, 0.16291192,
        0.17292513, 0.17305802, 0.17394781, 0.17408482, 0.17662939,
        0.18535793, 0.18537509, 0.18644536, 0.18645546, 0.19845716,
        0.19850652, 0.19941635, 0.19990181, 0.20266243, 0.20812335]),
 [((array([[ 9.54171367e-10,  2.48714699e-10,  4.22115874e-11, ...,
             2.30607906e-10, -1.32982223e-09, -7.66769448e-11],
           [ 1.04847518e-11, -1.10539340e-11,  4.10064429e-10, ...,
            -6.76497884e-10, -4.78101936e-11,  1.10853907e-11],
           [ 1.97255243e-07,  2.17359610e-07,  1.66871914e-08, ...,
             7.63388363e-09, -1.06969179e-07, -1.53678631e-08],
           ...,
           [ 1.91205786e-03, -1.48696822e-03, -2.90797679e-01, ...,
            -4.73049028e-05, -4.20241642e-06, -4.97373102e-07],
           [-3.18354688e-01,  2.60630137e-01, 

In [7]:

# ── 4. Oscillator strengths and spectrum ─────────────────────────────────────
td.analyze()    # prints TDM, prints dominant single excitations

e_exc_ev = np.array(td.e) * HA2EV          # excitation energies in eV
osc      = np.array(td.oscillator_strength())

# Save table
with open("tddft_roots.txt", "w") as out:
    out.write(f"{'Root':>6}  {'ΔE (eV)':>10}  {'f':>12}\n")
    out.write("-" * 34 + "\n")
    for i, (e, f) in enumerate(zip(e_exc_ev, osc)):
        out.write(f"{i+1:>6}  {e:>10.4f}  {f:>12.6f}\n")

print("\nTDDFT roots written to tddft_roots.txt")



** Excitation energies and oscillator strengths **
Excited State   1:    A      2.03077 eV    610.53 nm  f=0.0015
Excited State   2:    A      2.03421 eV    609.49 nm  f=0.0015
Excited State   3:    A      2.17042 eV    571.24 nm  f=0.0000
Excited State   4:    A      2.17350 eV    570.44 nm  f=0.0000
Excited State   5:    A      2.41205 eV    514.02 nm  f=0.0000
Excited State   6:    A      2.41378 eV    513.65 nm  f=0.0000
Excited State   7:    A      2.74326 eV    451.96 nm  f=0.0000
Excited State   8:    A      2.74358 eV    451.91 nm  f=0.0000
Excited State   9:    A      3.07264 eV    403.51 nm  f=0.0000
Excited State  10:    A      3.07548 eV    403.14 nm  f=0.0000
Excited State  11:    A      4.02695 eV    307.89 nm  f=0.0000
Excited State  12:    A      4.05276 eV    305.93 nm  f=0.0000
Excited State  13:    A      4.38988 eV    282.43 nm  f=0.0000
Excited State  14:    A      4.41860 eV    280.60 nm  f=0.0000
Excited State  15:    A      4.43306 eV    279.68 nm  f=0.0000
Exc

In [8]:

# ── 5. Broadened spectrum (Lorentzian, σ = 0.14 eV ≈ 0.005 Ha) ──────────────
sigma_ev = 0.005 * HA2EV   # match the σ used in TPSCI absorption_spectrum
e_grid   = np.linspace(0.5, 8.0, 2000)

def lorentzian(x, x0, sigma):
    return (sigma / np.pi) / ((x - x0)**2 + sigma**2)

spectrum = sum(f * lorentzian(e_grid, e0, sigma_ev)
               for e0, f in zip(e_exc_ev, osc) if f > 1e-5)


In [9]:

# ── 6. Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(e_grid, spectrum, lw=2, color="darkorange", label="TD-CAM-B3LYP/def2-SVP")

# sticks
for e, f in zip(e_exc_ev, osc):
    if f > 1e-4:
        ax.plot([e, e], [0, f * max(spectrum) / (max(osc) + 1e-10)],
                lw=1.5, color="saddlebrown", alpha=0.7)

ax.set_xlabel("Energy (eV)", fontsize=12)
ax.set_ylabel("Absorption (arb. u.)", fontsize=12)
ax.set_title("TDDFT absorption spectrum — Cr₂O(NH₃)₈⁴⁺", fontsize=12)
ax.legend()
ax.set_xlim(e_grid[0], e_grid[-1])
fig.tight_layout()
fig.savefig("tddft_spectrum.png", dpi=150)
print("Spectrum saved to tddft_spectrum.png")

Spectrum saved to tddft_spectrum.png
